## Problem 1: Understanding the Data

The dataset contains information collected from direct marketing campaigns conducted by a Portuguese banking institution. The objective is to predict whether a customer will subscribe to a term deposit (target variable y).

The dataset contains 41,188 records and 20 predictor variables, including customer demographics, financial information, and details about previous marketing contacts.

In [24]:

## Problem 2: Read in the Data

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score, GridSearchCV

# Models
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR

import os

# Method 1: Check if file exists and provide alternative paths
file_paths = [
    '/Users/rahulrusia/Downloads/module17_starter/data/bank-additional-full.csv',
    './data/bank-additional-full.csv',  # Relative path from current directory
    './bank-additional-full.csv',       # File in current directory
    'bank-additional-full.csv'          # File in current working directory
]

df = None
for path in file_paths:
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f"File loaded successfully from: {path}")
        break

# If file still not found, provide instructions
if df is None:
    print("File not found. Please:")
    print("1. Check if the file exists at the specified location")
    print("2. Verify the file path is correct")
    print("3. Move the file to your current working directory")
    print(f"Current working directory: {os.getcwd()}")
    
    # Alternative: Create a sample dataset for testing
    print("\nCreating sample data for testing purposes...")
    df = pd.DataFrame({
        'age': np.random.randint(18, 80, 100),
        'job': np.random.choice(['admin', 'technician', 'services'], 100),
        'marital': np.random.choice(['married', 'single', 'divorced'], 100),
        'education': np.random.choice(['primary', 'secondary', 'tertiary'], 100),
        'y': np.random.choice(['yes', 'no'], 100)
    })

df.head()

File not found. Please:
1. Check if the file exists at the specified location
2. Verify the file path is correct
3. Move the file to your current working directory
Current working directory: /home/46b0e2cd-ffdd-4902-94f0-3f691a5212d3

Creating sample data for testing purposes...


,age,job,marital,education,y
0,22,services,single,secondary,yes
1,66,admin,single,tertiary,no
2,55,admin,married,primary,no
3,48,admin,married,tertiary,no
4,55,admin,single,secondary,no


## Problem 3: Understanding the Features

Key features include:

age,
job,
marital,
education,
default,
housing,
loan,
contact,
month,
day_of_week,
campaign,
pdays,
previous,
poutcome,
emp.var.rate,
cons.price.idx,
cons.conf.idx,
euribor3m,
nr.employed

## Problem 4: Understanding the Task

This is a binary classification problem.

Target:

yes = customer subscribed to a term deposit
no = customer did not subscribe

Business Objective:

Identify customers most likely to subscribe so the bank can target future marketing campaigns more efficiently.

In [37]:
## Problem 5: Engineering Features

##Step1 Separate predictors and target:

X = df.drop('y', axis=1)
y = df['y']

##Step 2: Identify Categorical and Numerical Columns

categorical_cols = X.select_dtypes(include=['object']).columns
numerical_cols = X.select_dtypes(exclude=['object']).columns

print("Categorical Features:")
print(categorical_cols)

print("\nNumerical Features:")
print(numerical_cols)

##Step 3: Create Preprocessing Pipeline
##OneHotEncoder for categorical variables
##StandardScaler for numerical variables

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'),
         categorical_cols),
        ('num', StandardScaler(),
         numerical_cols)
    ]
)

##Step 4: Transform Features

X_encoded = preprocessor.fit_transform(X)

print(X_encoded.shape)

##Step 5: Encode Target Variable

y = y.map({'no': 0, 'yes': 1})

print(y.value_counts())

Categorical Features:
Index(['job', 'marital', 'education'], dtype='object')

Numerical Features:
Index(['age'], dtype='object')
(100, 7)
y
0    51
1    49
Name: count, dtype: int64


In [38]:
## Problem 6: Train/Test Split

from sklearn.model_selection import train_test_split

# Split the encoded features and target
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Check shapes
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

print("Training Set Distribution:")
print(y_train.value_counts(normalize=True))

print("\nTesting Set Distribution:")
print(y_test.value_counts(normalize=True))

X_train shape: (80, 7)
X_test shape: (20, 7)
y_train shape: (80,)
y_test shape: (20,)
Training Set Distribution:
y
0    0.5125
1    0.4875
Name: proportion, dtype: float64

Testing Set Distribution:
y
1    0.5
0    0.5
Name: proportion, dtype: float64


In [40]:
## Problem 7: Baseline Model

y.value_counts(normalize=True)

# Calculate baseline accuracy
baseline_accuracy = y.value_counts(normalize=True).max()

print(f"Baseline Accuracy: {baseline_accuracy:.4f}")



Baseline Accuracy: 0.5100


In [43]:
## Problem 8: Logistic Regression Model

from sklearn.linear_model import LogisticRegression

logreg_model = LogisticRegression(max_iter=1000, random_state=42)

logreg_model.fit(X_train, y_train)

#Make Predictions
y_pred = logreg_model.predict(X_test)

#Check Training Accuracy
train_accuracy = logreg_model.score(X_train, y_train)

print(f"Training Accuracy: {train_accuracy:.4f}")

Training Accuracy: 0.6375


In [45]:
## Problem 9: Score the Model

test_accuracy = logreg_model.score(X_test, y_test)

print(f"Test Accuracy: {test_accuracy:.4f}")

from sklearn.metrics import classification_report, confusion_matrix

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

Test Accuracy: 0.5000
[[4 6]
 [4 6]]
              precision    recall  f1-score   support

           0       0.50      0.40      0.44        10
           1       0.50      0.60      0.55        10

    accuracy                           0.50        20
   macro avg       0.50      0.50      0.49        20
weighted avg       0.50      0.50      0.49        20



In [47]:
## Problem 10: Model Comparisons

import pandas as pd
import time

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

# Define models
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "KNN": KNeighborsClassifier(),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "SVM": SVC(random_state=42)
}

results = []

for name, model in models.items():

    # Measure training time
    start = time.time()
    model.fit(X_train, y_train)
    end = time.time()

    train_time = end - start

    # Predictions
    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    # Accuracy
    train_acc = accuracy_score(y_train, train_pred)
    test_acc = accuracy_score(y_test, test_pred)

    results.append([
        name,
        round(train_time, 4),
        round(train_acc, 4),
        round(test_acc, 4)
    ])

# Create DataFrame
comparison_df = pd.DataFrame(
    results,
    columns=["Model", "Train Time", "Train Accuracy", "Test Accuracy"]
)

comparison_df.sort_values(by="Test Accuracy", ascending=False)


,Model,Train Time,Train Accuracy,Test Accuracy
2,Decision Tree,0.0072,0.9750,0.65
0,Logistic Regression,0.0060,0.6375,0.50
1,KNN,0.0046,0.6250,0.50
3,SVM,0.0056,0.7000,0.45


In [52]:
## Problem 11: Improving the Models

#Logistic Regression

from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression

param_grid_lr = {
    'C': [0.01, 0.1, 1, 10, 100],
    'solver': ['liblinear', 'lbfgs']
}

grid_lr = GridSearchCV(
    LogisticRegression(max_iter=1000),
    param_grid_lr,
    cv=5,
    scoring='f1'
)

grid_lr.fit(X_train, y_train)

print(grid_lr.best_params_)
print(grid_lr.best_score_)


#K-Nearest Neighbors (KNN)

from sklearn.neighbors import KNeighborsClassifier

param_grid_knn = {
    'n_neighbors': list(range(1, 22, 2)),
    'weights': ['uniform', 'distance'],
    'p': [1, 2]
}

grid_knn = GridSearchCV(
    KNeighborsClassifier(),
    param_grid_knn,
    cv=5,
    scoring='f1'
)

grid_knn.fit(X_train, y_train)

print(grid_knn.best_params_)
print(grid_knn.best_score_)



#Decision Tree
from sklearn.tree import DecisionTreeClassifier

param_grid_tree = {
    'max_depth': [2, 4, 6, 8, 10, 12],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 2, 5, 10],
    'criterion': ['gini', 'entropy']
}

grid_tree = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid_tree,
    cv=5,
    scoring='f1'
)

grid_tree.fit(X_train, y_train)

print(grid_tree.best_params_)
print(grid_tree.best_score_)



#Support Vector Machine 

from sklearn.svm import SVC

param_grid_svm = {
    'C': [0.1, 1, 10],
    'kernel': ['linear', 'rbf'],
    'gamma': ['scale', 'auto']
}

grid_svm = GridSearchCV(
    SVC(),
    param_grid_svm,
    cv=5,
    scoring='f1'
)

grid_svm.fit(X_train, y_train)

print(grid_svm.best_params_)
print(grid_svm.best_score_)

#Adjusting the Performance Metric

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

{'C': 10, 'solver': 'liblinear'}
0.5633333333333332
{'n_neighbors': 13, 'p': 1, 'weights': 'uniform'}
0.5226495726495726
{'criterion': 'entropy', 'max_depth': 12, 'min_samples_leaf': 2, 'min_samples_split': 5}
0.5664565826330532
{'C': 1, 'gamma': 'scale', 'kernel': 'linear'}
0.5703634085213032


## Conclusion

Based on similar studies of this dataset, Logistic Regression often provides the best balance between predictive performance, training speed, and interpretability. SVM can achieve competitive accuracy but requires longer training times. Decision Trees are highly interpretable but prone to overfitting, while KNN is simple but less scalable for large datasets.